# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ART001-coder/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this lane because the starter data already shows two things a flat rule would get
wrong: (1) expected CTR depends heavily on position tier — pages on `page_1` average roughly
8x the CTR of `deep` pages, so any single global "low CTR" cutoff will systematically mislabel
whole tiers — and (2) there is a large, review-sized pool of pages that already rank
(`avg_position <= 20`) and already get real traffic (`impressions_90d >= 500`) but still sit
below a reasonable CTR bar. That combination — real exposure, a measurable gap versus peers,
and enough volume per client to matter — is exactly what an opportunity-scoring queue needs.
I'm not choosing Lane 2 (general refresh scoring) because that lane's target (`trend_direction`)
mixes several different problems (visibility loss, engagement loss, staleness) into one label;
Lane 4 lets me isolate one clean, position-adjusted signal and test whether it beats a naive
rule. I'll confirm or swap this by the end of Week 4 once the signal audit is done.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Starter dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

# Does position tier actually change expected CTR? (mean CTR by tier, among pages with
# enough exposure to trust the number)
valid = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)]
ctr_by_tier = valid.groupby("position_tier")["ctr"].agg(["mean", "median", "count"]).sort_values("mean")
print("\nMean CTR (%) by position tier, impressions_90d >= 500, valid position:")
print(ctr_by_tier)


Starter dataset shape: (30000, 44)
Unique clients: 32

Mean CTR (%) by position tier, impressions_90d >= 500, valid position:
                   mean  median  count
position_tier                         
deep           0.043213    0.00    389
page_3_5       0.143236    0.09   4330
striking       0.266798    0.17   4485
page_1         0.338808    0.24   7064
top_3          0.346572    0.20    458


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages that already rank and already get exposure, which ones are
under-capturing clicks or engagement relative to *peers at the same position tier* — and should
therefore be reviewed first?

**Unit of analysis:** one content item (`content_id`) within one client (`client_id`), summarized
over its trailing 90-day window — not a query, not a day, not a client as a whole.

**Decision it improves:** which pages a content reviewer looks at *first* this cycle, out of far
more candidates than they have hours for.

**Who acts, and what they do:** a FlyRank content strategist / reviewer with limited weekly review
time across many client accounts. Given a ranked queue with reason codes, they open the top pages
and take one of a small set of actions: rewrite title/meta, improve snippet or schema, improve
on-page engagement (intro, structure, CTAs), or — if the page checks out on inspection — leave it
alone and move to the next.

**Cost of a wrong call:** this is a ranking/priority problem, not a yes/no classifier, so the real
cost is *mis-ordering*, not a single false positive or negative in isolation.
- Ranking a fine page too high wastes a reviewer's limited hour on a page that didn't need it —
  and that hour is drawn away from a page that did.
- Ranking a genuinely under-performing, high-exposure page too low means it keeps leaking clicks
  or engagement for another review cycle, which is real, uncaptured traffic for the client.
Because reviewer time is the scarce resource, the top of the queue has to be trustworthy — a
handful of position-1 pages with a small, real CTR gap can matter more than a long tail of noisy
`deep`-tier pages with a big percentage gap on tiny volume.

**Why data/ML helps at all:** a flat rule ("flag if CTR < 0.5%") ignores position tier entirely.
The numbers in Section 3 show mean CTR moves roughly 8x from the weakest tier to the strongest —
so a single global threshold would over-flag deep, high-authority-unlikely pages and under-flag
page-1 laggards that are quietly losing clicks. The fix doesn't require deep learning; it requires
*comparing each page only to its own tier* (expected-CTR-by-tier, then a gap/residual score) —
something a spreadsheet formula could technically do, but doing it well across tiers, intent
types, and volume levels, with a defensible ranked output and reason codes, is where a light model
or a carefully built composite score earns its place over a single if-statement.


In [2]:
# How many clients would actually see candidates in a review queue, and how lumpy is
# the candidate load across them? (grounds "limited reviewer capacity" above)
candidates = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
]
print("Candidate pages (impressions>=500, position<=20, ctr<0.5%):", len(candidates))
print("Clients with at least one candidate:", candidates["client_id"].nunique(), "of", df["client_id"].nunique())
print("\nCandidates per client (distribution):")
print(candidates.groupby("client_id").size().describe())


Candidate pages (impressions>=500, position<=20, ctr<0.5%): 9759
Clients with at least one candidate: 27 of 32

Candidates per client (distribution):
count      27.000000
mean      361.444444
std       776.597923
min         1.000000
25%        13.500000
50%        82.000000
75%       363.000000
max      3858.000000
dtype: float64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# 2-3 real numbers backing the lane choice

n_total = len(df)
valid_pos = df[df["avg_position"] > 0]
sub = valid_pos[valid_pos["impressions_90d"] >= 500]

# 1) Position tier changes expected CTR a lot -> a flat CTR threshold would be wrong
tier_means = sub.groupby("position_tier")["ctr"].mean().sort_values()
print(f"1) Mean CTR ranges from {tier_means.min():.3f}% ({tier_means.idxmin()} tier) "
      f"to {tier_means.max():.3f}% ({tier_means.idxmax()} tier) — "
      f"a {tier_means.max()/tier_means.min():.1f}x spread across position tiers.")

# 2) There is a sizeable, review-sized candidate pool
low_ctr = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
             & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)]
print(f"2) {len(low_ctr):,} of {n_total:,} pages ({100*len(low_ctr)/n_total:.1f}%) already rank "
      f"(position <= 20), already get real exposure (impressions_90d >= 500), "
      f"and still sit under a 0.5% CTR bar.")

# 3) The candidate pool touches most clients, but load is uneven -> a ranked queue (not a
#    flat list) is needed to spend limited reviewer time well
n_clients_affected = low_ctr["client_id"].nunique()
n_clients_total = df["client_id"].nunique()
print(f"3) These candidates span {n_clients_affected} of {n_clients_total} clients, but load per "
      f"client ranges from 1 page to {low_ctr.groupby('client_id').size().max():,} pages — "
      f"evidence a per-client, ranked queue is needed rather than a single global cutoff.")


1) Mean CTR ranges from 0.043% (deep tier) to 0.347% (top_3 tier) — a 8.0x spread across position tiers.
2) 9,759 of 30,000 pages (32.5%) already rank (position <= 20), already get real exposure (impressions_90d >= 500), and still sit under a 0.5% CTR bar.
3) These candidates span 27 of 32 clients, but load per client ranges from 1 page to 3,858 pages — evidence a per-client, ranked queue is needed rather than a single global cutoff.


## 4. Careful words: what I can and can't claim

**What I can claim, if the work holds up:**
- An *observed*, position-tier-adjusted association between certain page/content signals and
  under-capture of clicks or engagement, measured on this anonymized 30,000-row starter slice
  (and later the warehouse release).
- A *directional*, ranked list of review candidates with reason codes — "these pages look more
  worth a reviewer's time than those," not a certainty ranking.
- A *decision-support* tool: it orders limited reviewer attention. It does not replace reviewer
  judgment, and every recommendation should be inspectable (why did this page score high?).

**What I will never claim:**
- That rewriting a title/meta or improving engagement *causes* CTR or engagement to rise. I have
  no experiment (no A/B test, no before/after with a control) — only observational, historical
  data. Any recovery I later see in the data is a correlation candidate, not proof of causation.
- That I've discovered or reverse-engineered anything about Google's (or any AI platform's)
  ranking algorithm. CTR gaps versus tier peers are a content/searcher-behavior signal, not a
  ranking-factor claim.
- That a low score means "bad content" — low CTR at a given position can also reflect intent
  mismatch, snippet competition, or plain low search volume (noise), which I'll rule out with
  minimum-volume filters before trusting any single page's score.
- That this pool is complete or unbiased — it's 30,000 anonymized rows from 32 clients, not the
  full ~79M-row warehouse, so absolute counts here are illustrative of the *pattern*, not a
  claim about FlyRank's whole client base.


In [4]:
# Sanity check behind the "rule out noise" claim above: without a minimum-impression
# filter, "low CTR" candidates explode with tiny, unreliable samples.
no_filter = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)]
with_filter = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
                  & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)]
print(f"Candidates with NO minimum-impression filter: {len(no_filter):,}")
print(f"Candidates with impressions_90d >= 500 filter: {len(with_filter):,}")
print("Dropping the volume filter roughly "
      f"{len(no_filter)/max(len(with_filter),1):.1f}x's the pool — mostly noise from "
      "low-exposure pages, which is why the filter stays in every version of the queue.")


Candidates with NO minimum-impression filter: 16,558
Candidates with impressions_90d >= 500 filter: 9,759
Dropping the volume filter roughly 1.7x's the pool — mostly noise from low-exposure pages, which is why the filter stays in every version of the queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.